[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/36_int8_quantization.ipynb)

# 🔴 困难: INT8 量化线性层

实现一个**训练后量化（PTQ - Post-Training Quantization）的线性层**，使用 INT8 权重。

#### 函数签名
```python
class Int8Linear(nn.Module):
    def __init__(self, weight: Tensor, bias: Tensor = None): ...
    def forward(self, x: Tensor) -> Tensor: ...
```

#### 量化（按通道）
1. `scale = weight.abs().max(dim=1) / 127`
2. `weight_int8 = round(weight / scale).clamp(-128, 127).to(int8)`
3. 存储为 `register_buffer`（不可训练）
4. 前向：反量化（`int8.float() * scale`）然后矩阵乘法

In [ ]:
# 在 Colab 中安装 torch-judge（在 JupyterLab/Docker 中无操作）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✏️ 在此实现你的代码

class Int8Linear(nn.Module):
    def __init__(self, weight, bias=None):
        super().__init__()
        pass  # 量化权重, 注册 buffers

    def forward(self, x):
        pass  # 反量化和矩阵乘法

#### PTQ vs AWQ vs GPTQ 详细对比

复杂度和计算成本逐步递增

---

#### 1. PTQ (Post-Training Quantization) - 训练后量化

- 定义
训练完成后，直接对模型权重进行量化，**不需要额外训练或微调**。

- 方法分类

**最简单的PTQ：**
```python
# Min-Max量化
scale = (weight.max() - weight.min()) / 255
# 或
scale = weight.abs().max() / 127  # 对称量化
```

**稍微复杂点的PTQ：**
```python
# 使用KL散度寻找最优阈值（如TensorRT的做法）
# 在验证集上运行，统计激活值分布，找到最佳截断点
best_threshold = find_kl_divergence_threshold(activation_distribution)
```

优点: 不需要数据,几秒钟完成,实现简单。
缺点: 精度损失较大,没有考虑激活值分布。

---

#### 2. AWQ (Activation-aware Weight Quantization，感知激活值的权重量化) - 2023

- 核心思想
**"不是所有权重都同等重要，应该保护对激活值敏感的通道"**

- 关键洞察
通过分析**激活值的分布**，找出哪些权重通道对输出影响最大，为这些通道保留更高精度。

- 工作原理
```python
# 伪代码
def awq_quantize(weight, calibration_data):
    # 1. 用校准数据前向传播，收集激活值
    activations = []
    for batch in calibration_data:
        act = model(batch)  # 记录中间层激活值
        activations.append(act)
    
    # 2. 计算每个通道的重要性 = 激活值的平均绝对值
    importance = torch.stack(activations).abs().mean(dim=(0, 2, 3))
    
    # 3. 对重要通道，搜索最优缩放因子
    # 目的：最小化量化误差
    scales = []
    for channel_idx in range(num_channels):
        if importance[channel_idx] > threshold:
            # 重要通道：尝试多个缩放因子，选最优
            best_scale = search_scale(weight[:, channel_idx])
        else:
            best_scale = 1.0  # 不重要通道直接量化
        scales.append(best_scale)
    
    # 4. 应用缩放后量化
    weight_scaled = weight * scales
    scale = weight_scaled.abs().max(dim=1) / 127
    weight_int8 = round(weight_scaled / scale)
```

优点: 精度高,不需要训练,速度快，适用于 4-bit、3-bit 等低位宽。
缺点: 需要校准数据（~128-512个样本）

---

#### 3. GPTQ (Generative Pre-trained Transformer Quantization, 生成式预训练 Transformer 量化) - 2023

- 核心思想
**"逐层量化，并利用 Hessian 矩阵(海森矩阵)补偿量化误差"**\
Hessian 矩阵是一个二阶偏导数矩阵，用于描述多元函数的局部曲率。

- 关键洞察
量化某层的权重时，可以调整该层**未量化的权重**来补偿量化误差，使整体输出误差最小化。

- 工作原理
```python
# 伪代码
def gptq_quantize(layer_weight, calibration_data, bits=4):
    # 1. 收集激活值，计算 Hessian 矩阵（二阶梯度信息）
    H = compute_hessian(calibration_data, layer_weight)  # 重要！
    
    # 2. 逐行量化（或逐块量化）
    for row in range(layer_weight.shape[0]):
        # 量化当前行的权重
        weight_row = layer_weight[row]
        quantized_row = round(weight_row / scale) * scale
        
        # 3. 误差补偿：调整未量化的权重来补偿
        error = weight_row - quantized_row
        # 使用 Hessian 矩阵将误差分配到未量化的权重上
        compensation = solve_least_squares(H, error)
        # 更新未量化的权重
        layer_weight[row, :] += compensation
    
    return quantized_weight
```

#### 核心优势
- **误差补偿机制**：量化一部分权重时，调整其他权重来"弥补"
- **Hessian矩阵**：知道哪些权重调整后影响最小

优点: 精度非常高, 支持极低位宽,适用于大模型
缺点: 计算量大，需要 hessian 矩阵计算，实现复杂，需要校准数据。

---

- 实际应用示例

```python
# 1. PTQ - 最简单
model = torch.load('model.pth')
quantized = torch.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)

# 2. AWQ - 精度和速度的平衡
from awq import AutoAWQForCausalLM
model = AutoAWQForCausalLM.from_pretrained("llama-7b")
model.quantize(calibration_data, bits=4)  # 需要校准数据

# 3. GPTQ - 最高精度
from transformers import AutoModelForCausalLM, GPTQConfig
quantization_config = GPTQConfig(bits=4, dataset="c4")
model = AutoModelForCausalLM.from_pretrained("llama-7b", quantization_config=quantization_config)
```

---

- 选择建议

| 需求 | 推荐方法 |
|------|---------|
| 快速验证/原型 | PTQ |
| 生产部署，精度要求高 | AWQ |
| 追求极致精度（~FP16） | GPTQ |
| 小模型（<1B参数） | PTQ足够 |
| 大模型（>7B参数） | AWQ或GPTQ |
| 受限硬件（2-bit） | GPTQ |


要点：

1. **按通道量化**：每个输出通道有独立的`scale`，通过`weight.abs().max(dim=1)`计算该通道的最大绝对值

2. **量化过程**：
   - `weight / scale` 将权重归一化到[-127, 127]
   - `round` 四舍五入
   - `clamp(-128, 127)` 确保在int8范围内
   - 转换为`torch.int8`类型

3. **存储为buffer**：使用`register_buffer`确保这些张量随模型保存和加载，但不会被优化器更新

4. **前向推理**：
   - 反量化：`int8.float() * scale` 恢复为fp32
   - 使用标准的矩阵乘法
   - 添加偏置（保持fp32）

5. **偏置处理**：偏置不量化，直接存储为fp32，因为偏置通常占比较小

这种训练后量化方法能有效减少模型大小（权重从fp32降到int8，减少75%），同时保持较高的精度。

In [ ]:
from typing import Optional

class Int8Linear(nn.Module):
    def __init__(self, weight: torch.Tensor, bias: Optional[torch.Tensor] = None):
        """
        训练后量化的线性层，使用INT8权重,将权重张量使用 INT8 进行存储
        
        Args:
            weight: 原始浮点权重，shape (out_features, in_features)
            bias: 偏置项，shape (out_features,)
        """
        super().__init__()
        
        # 保存原始shape信息
        self.out_features, self.in_features = weight.shape
        
        # 按通道量化（每个输出通道独立量化）
        # scale shape: (out_features, 1)
        self.register_buffer('scale', weight.abs().max(dim=1, keepdim=True)[0] / 127)
        
        # 量化权重到int8
        # 注意：需要先转换为 float 再计算，最后转为int8
        weight_int8 = torch.round(weight / self.scale)
        weight_int8 = weight_int8.clamp(-128, 127).to(torch.int8)
        self.register_buffer('weight_int8', weight_int8)
        
        # 偏置保持为fp32，直接存储
        if bias is not None:
            self.register_buffer('bias', bias.clone())
        else:
            self.register_buffer('bias', None)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        前向传播：将 INT8 权重张量反量化然后矩阵乘法
        
        Args:
            x: 输入张量，shape (batch_size, in_features)
        
        Returns:
            输出张量，shape (batch_size, out_features)
        """
        # 反量化：int8 -> fp32 * scale
        # weight_fp32 shape: (out_features, in_features)
        weight_fp32 = self.weight_int8.float() * self.scale
        
        # 矩阵乘法
        out = torch.mm(x, weight_fp32.T)
        
        # 添加偏置
        if self.bias is not None:
            out = out + self.bias
        
        return out
    
    def extra_repr(self) -> str:
        """显示额外的信息"""
        return f'in_features={self.in_features}, out_features={self.out_features}'

In [ ]:
# 🧪 调试
w = torch.randn(8, 4)
q = Int8Linear(w)
x = torch.randn(2, 4)
print('输出:', q(x).形状)
print('dtype:', q.weight_int8.dtype)
print('最大量化误差:', (w - q.weight_int8.float() * q.scale).abs().max().item())

AWQ (Activation-aware Weight Quantization) 感知激活值的权重量化实现代码

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from typing import Optional, Tuple, List

class AWQQuantizer:
    """
    Activation-aware Weight Quantization
    核心思想：根据激活值重要性，保护重要通道
    """
    
    def __init__(self, bits: int = 4, group_size: int = 128, 
                 search_scale: bool = True, n_grid: int = 20):
        """
        Args:
            bits: 量化位数 (4, 8)
            group_size: 分组大小，None 表示 per-channel
            search_scale: 是否搜索最优缩放因子
            n_grid: 网格搜索的粒度
        """
        self.bits = bits
        self.group_size = group_size
        self.search_scale = search_scale
        self.n_grid = n_grid
        
    def quantize(self, model: nn.Module, calib_dataloader, 
                 num_samples: int = 128) -> nn.Module:
        """
        对模型进行AWQ量化
        
        Args:
            model: 待量化模型
            calib_dataloader: 校准数据加载器
            num_samples: 使用的校准样本数
        """
        # 1. 收集激活值统计信息
        act_stats = self._collect_activation_stats(model, calib_dataloader, num_samples)
        
        # 2. 逐层量化
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                self._quantize_layer(module, act_stats.get(name))
        
        return model
    
    def _collect_activation_stats(self, model: nn.Module, 
                                  dataloader, num_samples: int) -> dict:
        """
        收集每一层的激活值统计
        """
        act_stats = {}
        hooks = []
        
        def hook_fn(name):
            def hook(module, input, output):
                if name not in act_stats:
                    act_stats[name] = []
                # 只收集输入激活值
                if isinstance(input, tuple):
                    inp = input[0]
                else:
                    inp = input
                act_stats[name].append(inp.detach().cpu())
            return hook
        
        # 注册hooks
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                hooks.append(module.register_forward_hook(hook_fn(name)))
        
        # 收集激活值
        model.eval()
        with torch.no_grad():
            for i, batch in enumerate(dataloader):
                if i >= num_samples:
                    break
                if isinstance(batch, (list, tuple)):
                    batch = batch[0]
                model(batch)
        
        # 移除hooks
        for hook in hooks:
            hook.remove()
        
        # 统计激活值分布
        for name in act_stats:
            act_stats[name] = torch.cat(act_stats[name], dim=0)
            
        return act_stats
    
    def _quantize_layer(self, layer: nn.Linear, activations: Optional[torch.Tensor]):
        """
        量化单个线性层
        """
        if activations is None:
            # 如果没有激活值，使用普通量化
            self._quantize_layer_naive(layer)
            return
        
        weight = layer.weight.data
        out_features, in_features = weight.shape
        
        # 计算每个通道的重要性 = 激活值的平均绝对值
        # shape: (in_features,)
        importance = activations.abs().mean(dim=0)
        
        # 对权重进行分组
        if self.group_size is None:
            groups = [0, in_features]  # 整层量化
        else:
            groups = list(range(0, in_features, self.group_size))
            if groups[-1] < in_features:
                groups.append(in_features)
        
        # 逐组量化
        quantized_weight = torch.zeros_like(weight)
        scales = []
        
        for i in range(len(groups) - 1):
            start, end = groups[i], groups[i+1]
            weight_group = weight[:, start:end]
            importance_group = importance[start:end]
            
            # 计算该组的scale
            if self.search_scale and importance_group.sum() > 0:
                # AWQ的核心：搜索最优缩放因子
                best_scale = self._search_optimal_scale(
                    weight_group, importance_group, self.bits
                )
            else:
                # 普通量化
                best_scale = weight_group.abs().max().item() / (2**(self.bits-1) - 1)
            
            # 量化
            scale = best_scale
            max_val = 2**(self.bits-1) - 1
            weight_quant = torch.round(weight_group / scale).clamp(-max_val-1, max_val)
            
            quantized_weight[:, start:end] = weight_quant * scale
            scales.append(scale)
        
        # 更新权重
        layer.weight.data = quantized_weight
        
        # 存储scale用于推理
        if not hasattr(layer, 'awq_scales'):
            layer.awq_scales = []
        layer.awq_scales.extend(scales)
        
        # 将权重转换为int8存储（可选）
        layer.weight_int8 = torch.round(weight / layer.awq_scales[0]).to(torch.int8)
        
    def _search_optimal_scale(self, weight: torch.Tensor, 
                              importance: torch.Tensor, 
                              bits: int) -> float:
        """
        搜索最优缩放因子（AWQ的核心算法）
        
        通过网格搜索找到使量化误差最小的缩放因子
        """
        # 基础scale
        base_scale = weight.abs().max().item() / (2**(bits-1) - 1)
        
        # 网格搜索范围 [0.5, 2.0] * base_scale
        best_scale = base_scale
        best_error = float('inf')
        
        # 搜索空间
        search_scales = np.linspace(0.5, 2.0, self.n_grid) * base_scale
        
        max_val = 2**(bits-1) - 1
        
        for scale in search_scales:
            # 量化
            weight_quant = torch.round(weight / scale).clamp(-max_val-1, max_val)
            weight_dequant = weight_quant * scale
            
            # 计算加权误差（重要通道误差权重大）
            error = ((weight - weight_dequant) ** 2 * importance.unsqueeze(0)).sum()
            
            if error < best_error:
                best_error = error
                best_scale = scale
        
        return best_scale
    
    def _quantize_layer_naive(self, layer: nn.Linear):
        """普通量化（无激活值信息）"""
        weight = layer.weight.data
        max_val = 2**(self.bits-1) - 1
        scale = weight.abs().max().item() / max_val
        weight_quant = torch.round(weight / scale).clamp(-max_val-1, max_val)
        layer.weight.data = weight_quant * scale
        layer.awq_scales = [scale]

GPTQ (Generative Pre-trained Transformer Quantization, 生成式预训练 Transformer 量化) 实现代码

In [ ]:
class GPTQQuantizer:
    """
    GPTQ: Generative Pre-trained Transformer Quantization
    核心思想：使用Hessian矩阵补偿量化误差
    """
    
    def __init__(self, bits: int = 4, group_size: int = 128, 
                 damp_percent: float = 0.01, desc_act: bool = False):
        """
        Args:
            bits: 量化位数
            group_size: 分组大小
            damp_percent: Hessian矩阵正则化系数
            desc_act: 是否按列下降顺序量化
        """
        self.bits = bits
        self.group_size = group_size
        self.damp_percent = damp_percent
        self.desc_act = desc_act
        
    def quantize(self, model: nn.Module, calib_dataloader, 
                 num_samples: int = 128) -> nn.Module:
        """
        对模型进行GPTQ量化
        """
        # 收集每一层的输入
        layer_inputs = self._collect_layer_inputs(model, calib_dataloader, num_samples)
        
        # 逐层量化
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                inputs = layer_inputs.get(name)
                if inputs is not None:
                    self._quantize_layer_gptq(module, inputs)
        
        return model
    
    def _collect_layer_inputs(self, model: nn.Module, 
                              dataloader, num_samples: int) -> dict:
        """
        收集每一层的输入特征（用于计算Hessian）
        """
        layer_inputs = {}
        hooks = []
        
        def hook_fn(name):
            def hook(module, input, output):
                if name not in layer_inputs:
                    layer_inputs[name] = []
                if isinstance(input, tuple):
                    inp = input[0]
                else:
                    inp = input
                layer_inputs[name].append(inp.detach().cpu())
            return hook
        
        # 注册hooks
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                hooks.append(module.register_forward_hook(hook_fn(name)))
        
        # 前向传播收集输入
        model.eval()
        with torch.no_grad():
            for i, batch in enumerate(dataloader):
                if i >= num_samples:
                    break
                if isinstance(batch, (list, tuple)):
                    batch = batch[0]
                model(batch)
        
        # 移除hooks
        for hook in hooks:
            hook.remove()
        
        # 合并输入
        for name in layer_inputs:
            layer_inputs[name] = torch.cat(layer_inputs[name], dim=0)
            
        return layer_inputs
    
    def _quantize_layer_gptq(self, layer: nn.Linear, inputs: torch.Tensor):
        """
        使用GPTQ量化单个线性层
        
        核心步骤：
        1. 计算Hessian矩阵 H = X^T @ X
        2. 逐列量化权重
        3. 使用Hessian的逆矩阵补偿量化误差
        """
        weight = layer.weight.data.clone()  # shape: (out_features, in_features)
        out_features, in_features = weight.shape
        
        # 1. 计算Hessian矩阵
        # H = X^T @ X，其中X是输入特征
        X = inputs.to(layer.weight.device)
        H = X.T @ X
        
        # 添加正则化（防止Hessian奇异）
        damp = self.damp_percent * torch.mean(torch.diag(H))
        H += damp * torch.eye(H.shape[0], device=H.device)
        
        # 2. 计算Hessian的逆（使用Cholesky分解，更快更稳定）
        try:
            # Cholesky分解要求矩阵正定
            chol = torch.linalg.cholesky(H)
            H_inv = torch.cholesky_inverse(chol)
        except:
            # 如果失败，使用通用逆
            H_inv = torch.linalg.pinv(H)
        
        # 3. 确定量化顺序
        if self.desc_act:
            # 按Hessian对角线（重要性）降序排列
            order = torch.argsort(torch.diag(H), descending=True)
        else:
            # 顺序量化
            order = torch.arange(in_features)
        
        # 4. 逐列量化
        group_size = self.group_size if self.group_size else in_features
        max_val = 2**(self.bits-1) - 1
        
        for start in range(0, in_features, group_size):
            end = min(start + group_size, in_features)
            current_group = torch.arange(start, end, device=H.device)
            
            # 获取当前组的Hessian子矩阵
            H_group = H[current_group][:, current_group]
            
            # 对当前组的每一列进行量化
            for i, idx in enumerate(current_group):
                # 获取当前列的权重
                w = weight[:, idx]
                
                # 计算scale
                scale = w.abs().max().item() / max_val
                if scale < 1e-6:
                    scale = 1.0
                
                # 量化
                w_quant = torch.round(w / scale).clamp(-max_val-1, max_val)
                w_dequant = w_quant * scale
                
                # 更新权重
                weight[:, idx] = w_dequant
                
                # 计算量化误差
                error = w - w_dequant
                
                # 使用Hessian补偿误差（GPTQ的核心！）
                if i < len(current_group) - 1:
                    # 获取当前Hessian行的逆矩阵信息
                    # 将误差分配到未量化的权重上
                    # 使用Hessian的逆矩阵的一行
                    H_inv_row = H_inv[idx, current_group[i+1:]]
                    compensation = (error.unsqueeze(1) * H_inv_row.unsqueeze(0)).sum(dim=1)
                    
                    # 更新未量化的权重
                    for j, next_idx in enumerate(current_group[i+1:]):
                        weight[:, next_idx] -= compensation
        
        # 5. 更新layer的权重
        layer.weight.data = weight
        
        # 存储量化信息用于推理
        if not hasattr(layer, 'gptq_scales'):
            layer.gptq_scales = []
        # 每组的scale
        for start in range(0, in_features, group_size):
            end = min(start + group_size, in_features)
            group_weight = weight[:, start:end]
            scale = group_weight.abs().max().item() / max_val
            layer.gptq_scales.append(scale)

    def _quantize_layer_gptq_chunked(self, layer: nn.Linear, inputs: torch.Tensor):
        """
        分块版本的GPTQ（处理大矩阵时更高效）
        逐行而不是逐列量化，减少内存使用
        """
        weight = layer.weight.data.clone()
        out_features, in_features = weight.shape
        
        X = inputs.to(layer.weight.device)
        H = X.T @ X
        
        damp = self.damp_percent * torch.mean(torch.diag(H))
        H += damp * torch.eye(H.shape[0], device=H.device)
        
        max_val = 2**(self.bits-1) - 1
        group_size = self.group_size if self.group_size else in_features
        
        # 逐行量化（更容易并行）
        for row in range(0, out_features, 32):  # 每次处理32行
            row_end = min(row + 32, out_features)
            weight_chunk = weight[row:row_end]
            
            for start in range(0, in_features, group_size):
                end = min(start + group_size, in_features)
                
                # 处理每一列
                for col in range(start, end):
                    w = weight_chunk[:, col]
                    scale = w.abs().max().item() / max_val
                    if scale < 1e-6:
                        scale = 1.0
                    
                    w_quant = torch.round(w / scale).clamp(-max_val-1, max_val)
                    w_dequant = w_quant * scale
                    weight[row:row_end, col] = w_dequant
        
        layer.weight.data = weight

In [ ]:
# ✅ 提交
from torch_judge import check
check('int8_quantization')